# Alpha Engine — Colab E2 / US87 Model Batch (owner-operated)

This is a **real multi-trial XGBoost validation worker**, not a browser research IDE, hosted scheduler, GPU guarantee or strategy auto-publisher.

**Default: does not mount Drive, copy private data, train, or upload results.** Run only after reviewing actual source-provider rights for private Google Colab processing, your chosen immutable v5 market data and the complete 40-character CI-green experiment revision.

Reuses existing Study Workspace, US87 daily prices and QQQ. A validation search is NOT an untouched final-test evaluation. US Alpha158, PIT and trade-readiness remain blocked/unproven. **Never run the older alpha_research_exchange_v1 notebook against the v5 ZIP.**

Scope & code: [Issue #1373](https://github.com/liuh886/alpha_engine/issues/1373), stacked [PR #1374](https://github.com/liuh886/alpha_engine/pull/1374).

In [ ]:
from pathlib import Path
import hashlib, json, re, shutil, subprocess, sys

# Configure paths in your *private* Drive. No folder IDs, tokens, or sensitive
# metadata should be committed to GitHub.
V5_DIR = Path('/content/drive/MyDrive/CHANGE_ME/private/input_packages/us87-cn130-20261103-local-v5')
BATCH_SPEC_FILE = Path('/content/drive/MyDrive/CHANGE_ME/private/us87_xgb_batch.json')
CLOUD_RIGHTS_RECORD = Path('/content/drive/MyDrive/CHANGE_ME/private/us87_cloud_rights.json')
CHECKPOINT_PARENT = Path('/content/drive/MyDrive/CHANGE_ME/private/incoming_unreviewed')

EXPECTED_MANIFEST_SHA = 'd9f4fb2ef7aa2735cfb95436e7eabb744ffb8d6d387c4294626893fc262dab0c'
EXPECTED_US_HISTORY_SHA = '93b2d42cebf3ebfac7b80298f6822fdc4305f7550b73ac1b2bb8d272d3fb3de8'
VERIFIER_AND_MODEL_RUNNER_GIT_SHA = 'CHANGE_ME_EXACT_PR1374_COMMIT_SHA'

OPERATOR_APPROVES_PRIVATE_CLOUD_DATA_AND_TRAINING = False
RUN_ONE_TRAINING_CHUNK = False
EXPORT_IMMUTABLE_CHECKPOINT = False
CHUNK_TRIAL_BUDGET = 2

# Optional exact folder from a previous immutable checkpoint; never scan
# unattended or automatically select a "latest" checkpoint.
RESUME_FROM_VERIFIED_CHECKPOINT = None

assert OPERATOR_APPROVES_PRIVATE_CLOUD_DATA_AND_TRAINING, 'Owner opt-in required'
assert all('CHANGE_ME' not in str(p) for p in
           (V5_DIR, BATCH_SPEC_FILE, CLOUD_RIGHTS_RECORD, CHECKPOINT_PARENT))
assert re.fullmatch(r'[a-f0-9]{40}', VERIFIER_AND_MODEL_RUNNER_GIT_SHA)
assert 1 <= CHUNK_TRIAL_BUDGET <= 20

## 1 · Rights review and private Drive mount

A provider-specific authorization record must be reviewed by the owner. Permission to mount Drive is **not** permission from Yahoo or any other upstream price-data source. If the intended cloud computation is not licensed, stop before mounting/copying and use local-only execution.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

rights = json.loads(CLOUD_RIGHTS_RECORD.read_text(encoding='utf-8'))
assert rights.get('schema_version') == 'e1a_price_cloud_rights_v1'
assert rights.get('market') == 'us'
assert rights.get('bundle_manifest_sha256') == EXPECTED_MANIFEST_SHA
assert rights.get('history_archive_sha256') == EXPECTED_US_HISTORY_SHA
assert rights.get('cloud_processing_purpose') == 'private_colab_daily_price_research'
assert rights.get('owner_approved_private_cloud_processing') is True
assert rights.get('provider_rights') and all(
    p.get('private_cloud_compute_permitted') is True and
    str(p.get('terms_url', '')).startswith('https://') and
    len(str(p.get('review_basis', '')).strip()) >= 12
    for p in rights['provider_rights']
)
print('Owner-attested rights record present (not independent vendor certification).')

## 2 · Copy exactly the authorized price input once, then pin code and data

In [ ]:
def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024*1024), b''):
            h.update(block)
    return h.hexdigest()

scratch = Path('/content/p2_e2_scratch')
scratch.mkdir(parents=True, exist_ok=False)
for src, name, expected in [
    (V5_DIR/'bundle-manifest.json', 'bundle-manifest.json', EXPECTED_MANIFEST_SHA),
    (V5_DIR/'us_market_history.zip', 'us_market_history.zip', EXPECTED_US_HISTORY_SHA)
]:
    dest = scratch/name
    shutil.copyfile(src, dest)
    assert sha256_file(dest) == expected, f'external byte pin mismatch: {name}'
batch_file = scratch/'batch.json'
shutil.copyfile(BATCH_SPEC_FILE, batch_file)
batch = json.loads(batch_file.read_text())
assert batch['runner_git_sha'] == VERIFIER_AND_MODEL_RUNNER_GIT_SHA
assert batch['v5_bundle_manifest_sha256'] == EXPECTED_MANIFEST_SHA
assert batch['history_archive_sha256'] == EXPECTED_US_HISTORY_SHA
assert batch['purpose'] == 'validation_search_only'

checkout = Path('/content/p2_e2_checkout')
subprocess.run(['git','clone','--quiet','--filter=blob:none','--no-checkout',
                'https://github.com/liuh886/alpha_engine.git',str(checkout)],check=True)
subprocess.run(['git','-C',str(checkout),'checkout','--detach',
                VERIFIER_AND_MODEL_RUNNER_GIT_SHA],check=True)
assert subprocess.check_output(['git','-C',str(checkout),'rev-parse','HEAD'],
                               text=True).strip() == VERIFIER_AND_MODEL_RUNNER_GIT_SHA
subprocess.run([sys.executable,'-m','pip','install','--quiet','uv'],check=True)
subprocess.run(['uv','sync','--frozen'],cwd=checkout,check=True)

## 3 · Verify v5 bytes and build an isolated, non-Git research data root

The original dataset remains immutable and local-only. Cloud permission is a separate private operator record, never a mutation to v5.

In [ ]:
def uv_module(*args):
    subprocess.run(['uv','run','--frozen','python','-m',*map(str,args)],
                   cwd=checkout,check=True)

uv_module('src.research.e1a_price_preflight',
          '--manifest',scratch/'bundle-manifest.json',
          '--archive',scratch/'us_market_history.zip',
          '--expected-manifest-sha256',EXPECTED_MANIFEST_SHA,
          '--expected-archive-sha256',EXPECTED_US_HISTORY_SHA,
          '--rights-record',CLOUD_RIGHTS_RECORD,'--require-cloud-rights')
stage_root = scratch/'isolated_us87'
uv_module('src.research.v5_isolated_staging','stage',
          '--manifest',scratch/'bundle-manifest.json',
          '--archive',scratch/'us_market_history.zip',
          '--expected-manifest-sha256',EXPECTED_MANIFEST_SHA,
          '--expected-archive-sha256',EXPECTED_US_HISTORY_SHA,
          '--output-root',stage_root)
uv_module('src.research.colab_experiment_batch','plan',
          '--batch-spec',batch_file)
print('Verified complete US87 input; no model execution yet.')

## 4 · Optional: restore one explicit, immutable prior checkpoint

Only import from a trusted private `incoming_unreviewed` folder after checking exact file hashes against its receipt. The local run identity, data and model Git SHA are still checked by the batch runner on `--resume`.

In [ ]:
output_root = scratch/'runs'
resume_flag = False
if RESUME_FROM_VERIFIED_CHECKPOINT is not None:
    saved = Path(RESUME_FROM_VERIFIED_CHECKPOINT)
    index = json.loads((saved/'checkpoint-index.json').read_text())
    assert index['batch_spec_sha256'] == sha256_file(batch_file)
    assert index['runner_git_sha'] == VERIFIER_AND_MODEL_RUNNER_GIT_SHA
    assert index['history_archive_sha256'] == EXPECTED_US_HISTORY_SHA
    assert not output_root.exists()
    output_root.mkdir()
    for item in index['files']:
        rel = Path(item['path'])
        assert not rel.is_absolute() and '..' not in rel.parts
        src = saved/rel
        dst = output_root/rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(src,dst)
        assert sha256_file(dst) == item['sha256']
    resume_flag = True
    print('Prior trial files restored with independent checkpoint hashes.')

## 5 · Run one bounded model-training chunk

Each invocation fits at most `CHUNK_TRIAL_BUDGET` new trials. Do not run this cell unless you have a provider-rights-approved cloud compute session and have inspected the frozen study spec. This is a validation search; no automatic final holdout or trading.

In [ ]:
if RUN_ONE_TRAINING_CHUNK:
    cmd = ['uv','run','--frozen','python','-m',
           'src.research.colab_experiment_batch','run',
           '--batch-spec',str(batch_file),'--data-root',str(stage_root),
           '--output-root',str(output_root),'--max-new-trials',
           str(CHUNK_TRIAL_BUDGET)]
    if resume_flag:
        cmd.append('--resume')
    subprocess.run(cmd,cwd=checkout,check=True)
    resume_flag = True  # Subsequent chunks reuse prior completed trials.
else:
    print('No model fit: RUN_ONE_TRAINING_CHUNK is False.')

## 6 · Save a hash-verified, *unreviewed* checkpoint to private Drive

Run this after each training chunk. The snapshot is never authoritative and never overwrites a prior checkpoint. It retains only frozen spec and research receipts, **not raw bars or credentials**. Private licensing/visibility remains owner-controlled; independently review the content before exporting.

In [ ]:
if EXPORT_IMMUTABLE_CHECKPOINT:
    assert RUN_ONE_TRAINING_CHUNK and output_root.is_dir()
    import datetime
    files = []
    for f in sorted(output_root.rglob('*')):
        if f.is_file():
            rel = f.relative_to(output_root).as_posix()
            assert '..' not in Path(rel).parts
            files.append({'path':rel,'sha256':sha256_file(f),'size':f.stat().st_size})
    assert files
    index = {
        'schema_version':'p2_e2_unreviewed_trial_checkpoint_v1',
        'batch_spec_sha256':sha256_file(batch_file),
        'runner_git_sha':VERIFIER_AND_MODEL_RUNNER_GIT_SHA,
        'history_archive_sha256':EXPECTED_US_HISTORY_SHA,
        'files':files,'research_only':True,
        'trade_ready':False,'pit_certified':False,
        'review_status':'incoming_unreviewed'}
    stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    dest = CHECKPOINT_PARENT/('e2-us87-checkpoint-'+stamp)
    dest.mkdir(parents=True,exist_ok=False)
    for row in files:
        from_file = output_root/row['path']
        to_file = dest/row['path']
        to_file.parent.mkdir(parents=True,exist_ok=True)
        shutil.copyfile(from_file,to_file)
        assert sha256_file(to_file) == row['sha256']
    (dest/'checkpoint-index.json').write_text(json.dumps(index,indent=2))
    print('Private unreviewed checkpoint saved; exact path:',dest)
else:
    print('No Drive checkpoint export. Local Colab VM scratch is ephemeral.')

## Acceptance notes

Do not claim Colab success until real model training, completed/blocked trial receipts, identical local baseline and actual resource observations exist. After the batch finishes, use the existing local Agent study review to inspect every run and cost assumption. Explicitly reserve a **separate, untouched final test** after hyperparameter selection. This notebook cannot certify licensing, point-in-time availability, survivorship-free universe selection or trade readiness.